# Pencarian audio: encoder Whisper + cosine
        **audio › retrieval** · notebook `whisper_encoder_cosine`

        **Kapan dipakai:** Embedding terkuat untuk ucapan di sini.

        **Jebakan umum:** Sama seperti di atas.

> ⚠️ Notebook ini berat (butuh torch / bobot / GPU). Lihat sel pertama.

**Data demo:** Speech Commands mini (8 kata: down/go/left/no/right/stop/up/yes; 1200 latih / 320 uji; 1 detik, 16 kHz). Galeri = klip latih; query = klip uji; **relevan** = kata yang sama.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi

In [ ]:
DATA_ROOT = ROOT / "data_latihan/audio/classification/speech_commands_mini"; FEAT = "whisper"; N_QUERY, TOPK, SEED = 200, 10, 42
MODEL_KEY = "whisper_encoder_cosine"; OUT = out_dir("audio", "retrieval", MODEL_KEY); set_seed(SEED)

## 2. Data dan embedding

In [ ]:
from superdb.audio import load_wavs, mfcc_stats, Wav2VecEmbedder, WhisperEmbedder
from sklearn.preprocessing import normalize, StandardScaler
tr = pd.read_csv(DATA_ROOT / "train.csv"); te = pd.read_csv(DATA_ROOT / "test.csv").merge(pd.read_csv(DATA_ROOT / "test_labels.csv"), on="id").sample(N_QUERY, random_state=SEED)
G, Q = load_wavs(tr["path"], DATA_ROOT, 16000), load_wavs(te["path"], DATA_ROOT, 16000); g_lab, q_lab = tr["label"].to_numpy(), te["label"].to_numpy()
if FEAT == "mfcc":
    sc = StandardScaler().fit(mfcc_stats(G)); Eg, Eq = normalize(sc.transform(mfcc_stats(G))), normalize(sc.transform(mfcc_stats(Q)))
else:
    emb = WhisperEmbedder(); Eg, Eq = normalize(emb.encode(G)), normalize(emb.encode(Q))
print(Eg.shape, Eq.shape)

## 3. Metrik ranking
Pembanding acak: P@10 ≈ proporsi kata (12,5% untuk 8 kata).

In [ ]:
from superdb.text_extra import retrieval_metrics
scores = Eq @ Eg.T; metrics = retrieval_metrics(scores, q_lab, g_lab, k=TOPK); display(show_table({**metrics, "acak P@k (dasar)": float(np.mean([np.mean(g_lab == l) for l in q_lab]))}))
save_json({"metrics": metrics, "feat": FEAT}, OUT / "metrics.json"); print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan MFCC, wav2vec2, Whisper pada query yang sama.
- Pencarian kata kunci dalam audio panjang: potong jadi jendela, embedding tiap jendela, cari yang terdekat dengan query.